# PetFinder – 3. mérföldkő: modellezés és kiértékelés

**Cél:** az M2-ben előkészített jellemzőkön modelleket tanítani, hangolni és összehasonlítani.

**Bemenet:** az M2 notebook `petfinder_m2_results` mappája: 50 jellemző és a `RescuerID` szerinti train / validation / belső test felosztás.

**Menet:** metrika → kiértékelési protokoll → baseline → további modellek → hangolás → végső összehasonlítás.

## 1. Beállítás és adatok betöltése

Az M2 kimenetét használjuk, így a szöveges és képi jellemzőket nem kell újra kiszámolni. Ha a mappa nincs meg, Colabban a cella bekéri a `petfinder_m2_results.zip` fájlt.

In [ ]:
import random, zipfile
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

try:
    from google.colab import files
    ROOT = Path("/content")
except ImportError:
    files = None
    ROOT = Path.cwd()

OUT = ROOT / "petfinder_m3_results"
OUT.mkdir(exist_ok=True)


def find_m2_results():
    # Előbb a megszokott helyeken keressük, csak utána kérünk feltöltést.
    for folder in (ROOT / "petfinder_m2_results", ROOT / "m2_upload"):
        hits = sorted(folder.rglob("train_features.csv")) if folder.exists() else []
        if hits:
            return hits[0].parent
    assert files is not None, "Hiányzik az M2 kimenete: futtasd előbb a PetFinder_M2 notebookot."
    print("Töltsd fel az M2 kimenetét ZIP-ként (petfinder_m2_results.zip):")
    uploaded = files.upload()
    name = next(n for n in uploaded if n.endswith(".zip"))
    with zipfile.ZipFile(ROOT / name) as z:
        z.extractall(ROOT / "m2_upload")
    return next((ROOT / "m2_upload").rglob("train_features.csv")).parent


M2 = find_m2_results()
print("M2 kimenet:", M2)

In [ ]:
TARGET = "AdoptionSpeed"
CATEGORIES = ["Type", "Breed1", "Breed2", "Gender", "Color1", "Color2",
              "Color3", "State", "MaturitySize", "FurLength", "Health",
              "Vaccinated", "Dewormed", "Sterilized"]

train = pd.read_csv(M2 / "train_features.csv")
val = pd.read_csv(M2 / "validation_features.csv")
test = pd.read_csv(M2 / "internal_test_features.csv")
manifest = pd.read_csv(M2 / "split_manifest.csv")

# A feature-fájlok sorrendje megegyezik a manifestével, így a RescuerID onnan jön.
g_train = manifest.loc[manifest.split == "train", "RescuerID"].to_numpy()
g_val = manifest.loc[manifest.split == "validation", "RescuerID"].to_numpy()
assert len(g_train) == len(train) and len(g_val) == len(val)
assert (manifest.split == "test").sum() == len(test)

# Fejlesztői halmaz = train + validation; a belső teszt külön marad.
dev = pd.concat([train, val], ignore_index=True)
X_dev, y_dev = dev.drop(columns=TARGET), dev[TARGET].to_numpy()
X_test, y_test = test.drop(columns=TARGET), test[TARGET].to_numpy()
groups = np.concatenate([g_train, g_val])

CAT_COLS = [c for c in CATEGORIES if c in X_dev.columns]
NUM_COLS = [c for c in X_dev.columns if c not in CAT_COLS]
for c in CAT_COLS:
    dtype = pd.CategoricalDtype(sorted(X_dev[c].dropna().unique()))
    X_dev[c], X_test[c] = X_dev[c].astype(dtype), X_test[c].astype(dtype)

print(f"Fejlesztői: {X_dev.shape}, belső teszt: {X_test.shape}")
print(f"Kategorikus: {len(CAT_COLS)}, numerikus: {len(NUM_COLS)}")
pd.Series(y_dev).value_counts(normalize=True).sort_index().round(3).rename("arány")

## 2. Kiértékelési metrikák

**Elsődleges metrika: QWK (quadratic weighted kappa)**
- Az `AdoptionSpeed` sorrendi (0–4). A QWK a tévedés távolságát négyzetesen bünteti: 0 helyett 4-et jósolni sokkal rosszabb, mint 0 helyett 1-et.
- A Kaggle-verseny hivatalos metrikája is ez, így az eredmény összevethető.
- 1 = tökéletes egyezés, 0 = véletlen szint. Ezt maximalizáljuk a modellválasztásnál és a hangolásnál.

**Kiegészítő metrikák**
- **Macro-F1:** minden osztály egyforma súllyal számít, így kiderül, ha a modell elhanyagolja a ritka 0-s osztályt (kb. 3%).
- **Accuracy:** csak tájékoztató, mert nem veszi figyelembe sem a sorrendet, sem az osztályok arányát.
- A végső modelleknél konfúziós mátrixot és osztályonkénti riportot is nézünk.

In [ ]:
from sklearn.metrics import cohen_kappa_score, f1_score, accuracy_score


def qwk(y_true, y_pred):
    return cohen_kappa_score(y_true, y_pred, weights="quadratic")


def scores(y_true, y_pred):
    return {"QWK": qwk(y_true, y_pred),
            "Macro-F1": f1_score(y_true, y_pred, average="macro", zero_division=0),
            "Accuracy": accuracy_score(y_true, y_pred)}


# Ellenőrzés: azonos hibaszám mellett a távolabbi tévedés kisebb QWK-t ad.
y = np.array([0, 1, 2, 3, 4] * 20)
near, far = y.copy(), y.copy()
near[:10] = np.clip(y[:10] + 1, 0, 4)
far[:10] = 4 - y[:10]
print(f"Közeli tévedések QWK: {qwk(y, near):.3f} | távoli tévedések QWK: {qwk(y, far):.3f}")

## 3. Kiértékelési protokoll

- A train + validation halmazon **5-szörös `StratifiedGroupKFold`** keresztvalidációt használunk, `RescuerID` szerint csoportosítva. Egy megmentő hirdetései hasonlóak, ezért mind ugyanabba a foldba kerülnek. Különben túl optimista lenne a becslés.
- Minden modell **ugyanazokat a foldokat** kapja, így az eredmények közvetlenül összevethetők.
- A **belső tesztet csak a végén, egyszer** használjuk a kiválasztott modellekre. Az M2 már mért rajta egy LightGBM-et, de semmit nem választottunk ki az alapján.

In [ ]:
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.base import clone

FOLDS = list(StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
             .split(X_dev, y_dev, groups))
for tr, va in FOLDS:
    assert not set(groups[tr]) & set(groups[va])

results, oof = {}, {}


def cross_validate(name, model):
    # Out-of-fold predikció: minden sort olyan modell jósol, amely nem látta.
    pred = np.zeros(len(y_dev), dtype=int)
    fold_scores = []
    for tr, va in FOLDS:
        m = clone(model).fit(X_dev.iloc[tr], y_dev[tr])
        pred[va] = m.predict(X_dev.iloc[va])
        fold_scores.append(qwk(y_dev[va], pred[va]))
    oof[name] = pred
    results[name] = {**scores(y_dev, pred), "QWK szórás (foldok)": np.std(fold_scores)}
    return pd.Series(results[name]).round(4)

## 4. Baseline modellek

A baseline megmutatja, mennyit ér egy egyszerű megoldás. Ehhez mérjük a későbbi modelleket.

1. **Leggyakoribb osztály:** mindig a 4-es osztályt jósolja. QWK-ja 0, ez az alsó határ.
2. **Logisztikus regresszió:** egyszerű, gyors, jól értelmezhető lineáris modell. Ha egy bonyolultabb modell nem veri jelentősen, nem éri meg a bonyolultságot.
   - Kategorikus oszlopok: one-hot kódolás. Numerikus oszlopok: medián pótlás és standardizálás.
   - `class_weight="balanced"`, hogy a ritka 0-s osztály se tűnjön el.

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression

linear_prep = ColumnTransformer([
    ("cat", make_pipeline(SimpleImputer(strategy="most_frequent"),
                          OneHotEncoder(handle_unknown="ignore")), CAT_COLS),
    ("num", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), NUM_COLS),
])

baselines = {
    "Leggyakoribb osztály": DummyClassifier(strategy="most_frequent"),
    "Logisztikus regresszió": make_pipeline(
        linear_prep, LogisticRegression(max_iter=2000, class_weight="balanced",
                                        random_state=SEED)),
}
for name, model in baselines.items():
    print(name, cross_validate(name, model).to_dict())

In [ ]:
summary = pd.DataFrame(results).T.round(4)
summary.to_csv(OUT / "cv_results.csv")
summary

**Értelmezés:** a leggyakoribb osztály QWK-ja 0, a logisztikus regresszió adja az első valódi viszonyítási pontot. A következő modelleknek ezt a QWK-t kell megverniük, ugyanezeken a foldokon.